# Part II — DeSE2026 revision: nested validation, paired bootstrap, full-information baseline, SMS-free model, repeated splits

This notebook reproduces every number in the revised conference paper *From Prediction to Action: Policy-Constrained Machine Learning for Lead Qualification Under Limited Sales Capacity* (DeSE2026, paper #1571378349). It is driven by `revision_experiments.py`, which re-implements the dissertation pipeline (Part I notebook, unchanged) and adds the analyses requested by the reviewers.

**Pinned versions.** XGBoost 2.1.4–3.2.0 reproduce the submitted numbers (265/270/266 converters) exactly; 3.3.0 and later do not (262/269/264) because of changes in tree-construction numerics. This notebook pins XGBoost 3.2.0.

**Runtime.** Steps 1–5 take about 5 minutes on 2 CPUs. Step 6 (25 independently tuned splits × 2 specifications) takes about 45 minutes; set `RUN_REPEATED = True` to recompute, otherwise the stored `results/repeated_splits.csv` is loaded.

In [ ]:
!pip install -q "xgboost==3.2.0" "scikit-learn==1.9.1" "pandas>=2.2" "numpy>=2" scipy
import sys, os, json, hashlib
import numpy as np, pandas as pd
REPO = "." if os.path.exists("revision_experiments.py") else "/content/drive/MyDrive/Masters_Constrained_Lead_Qualification"   # repo root, or Colab + Drive
DATA = f"{REPO}/Lead Scoring.csv" if os.path.exists(f"{REPO}/Lead Scoring.csv") else "/content/drive/MyDrive/Masters_Constrained_Lead_Qualification/Lead Scoring.csv"
sys.path.insert(0, REPO)
import revision_experiments as R
import xgboost, sklearn
print("xgboost", xgboost.__version__, "| scikit-learn", sklearn.__version__)
print("dataset sha256:", hashlib.sha256(open(DATA, "rb").read()).hexdigest())

## 1. Data and the SMS overlap (Review 1, point 4)

In [ ]:
df, policy = R.load_and_clean(DATA)                                   # notebook-identical cleaning (global >40% rule)
df_fold, policy_fold = R.load_and_clean(DATA, drop_high_missing_globally=False)   # fold-wise rule, used by all nested runs
assert df.shape == (7553, 28), df.shape
from sklearn.model_selection import train_test_split
y = df["Converted"]; X = df.drop(columns=["Converted"])
X_tr, X_te, y_tr, y_te, P_tr, P_te = train_test_split(X, y, policy, test_size=0.2, random_state=42, stratify=y)
for name, P_, X_ in [("train", P_tr, X_tr), ("test", P_te, X_te)]:
    ct = pd.crosstab(P_["Last Activity"].eq("SMS Sent"), X_["Last Notable Activity"].eq("SMS Sent"),
                     rownames=["Last Activity = SMS Sent (P2)"], colnames=["Last Notable Activity = SMS Sent (in model)"])
    print(f"\n{name}:\n{ct}\n  share of P2 leads carrying the leaked feature: {ct.loc[True, True] / ct.loc[True].sum():.3f}"
          f"\n  share of leaked-feature leads that are P2: {ct.loc[True, True] / ct[True].sum():.3f}")

## 2. Reproduction of the as-submitted configuration
Single 80/20 split, full-data cutoff, cap 40%, w 0.90, test-selected parameters. Expected under XGBoost ≤ 3.2.0: S1 265, S2 270, S3 266, test AUC 0.848.

In [ ]:
from sklearn.metrics import roc_auc_score
Xr_tr, Xr_te = R.feature_frame(X_tr, "orig"), R.feature_frame(X_te, "orig")
xgb0, params0 = R.tune_xgb(Xr_tr, y_tr)
res0 = P_te.reset_index(drop=True).copy(); res0["actual"] = y_te.values; res0["ml_score"] = xgb0.predict_proba(Xr_te)[:, 1]
sys0, diag0 = R.rank_systems(res0, ["Working Professional"], ["SMS Sent"], 0.40, 0.90, float(df["Converted"].mean()))
print("test AUC", round(roc_auc_score(y_te, res0["ml_score"]), 4), "| params", params0)
print("converters / Precision@K:", R.precision_at_k(sys0, diag0["K"]))
print("diagnostics:", diag0)

## 3. Nested 60/20/20 runs, primary split (seed 42), both specifications
Engine = validation-AUC winner of LR vs XGBoost; signals by the pre-stated rule on training rows; cap and weight on validation Precision@K; test evaluated once. `nosms` = primary specification (Last Notable Activity removed from the initial model); `orig` = original feature set.

In [ ]:
runs = {spec: R.nested_run(df_fold, policy_fold, 42, spec, tune=True) for spec in ["nosms", "orig"]}
keys = ["selected_engine", "val_auc_logit", "val_auc_xgb", "test_auc_engine", "val_auc_full", "test_auc_full", "p1", "p2",
        "p1_train_rate", "p1_train_n", "p2_train_rate", "p2_train_n", "cutoff", "cap", "blend_w", "fold_dropped_high_missing",
        "K", "S1_conv", "S2_conv", "S3_conv", "S4_conv", "S1_prec", "S2_prec", "S3_prec", "S4_prec", "diag_test", "cap_sens_test", "k_sens_test"]
for spec, r in runs.items():
    print(f"\n=== {spec} ===")
    for k in keys: print(f"  {k}: {r[k]}")
    print("  leakage audit:", r["leakage_audit"])

## 4. Paired bootstrap (Review 1 point 1, Review 2 point 3, Review 3 point 3)
Every system is applied to the same resample; the difference is computed within each resample. Intervals condition on the trained models and selected policies.

In [ ]:
pb = {}
for spec, r in runs.items():
    pb[spec], raw = R.paired_bootstrap(r["_res_te"], r["p1"], r["p2"], r["cap"], r["blend_w"], r["cutoff"], n=2000)
    rows = [{"difference": k.split("_")[0], "metric": k.split("_")[1], **v} for k, v in pb[spec].items() if "-" in k]
    print(f"\n=== {spec} ===\n", pd.DataFrame(rows).round(4).to_string(index=False))

## 5. Capacity and tier-cap sensitivity on the test partition (policy parameters held at validation-selected values)

In [ ]:
for spec, r in runs.items():
    print(f"\n=== {spec}: Precision@K by capacity ===")
    print(pd.DataFrame(r["k_sens_test"]).T.round(3))
    print(f"=== {spec}: S2 Precision@K by tier cap ===")
    print(pd.Series(r["cap_sens_test"]).round(3).to_string())

## 6. Repeated independently tuned splits (Review 1 point 5, Review 3 point 3)
Each split re-runs the hyperparameter searches, engine selection, signal selection and policy selection on its own training/validation rows.

In [ ]:
RUN_REPEATED = False
if RUN_REPEATED:
    rows = []
    for spec in ["orig", "nosms"]:
        for seed in range(25):
            r = R.nested_run(df_fold, policy_fold, seed, spec, tune=True)
            rows.append({k: r[k] for k in ["seed", "spec", "p1", "p2", "cap", "blend_w", "selected_engine", "test_auc_engine", "test_auc_full",
                                           "S1_conv", "S2_conv", "S3_conv", "S4_conv", "S1_prec", "S2_prec", "S3_prec", "S4_prec"]})
            print(seed, spec, r["S1_prec"], r["S2_prec"], r["S3_prec"], r["S4_prec"])
    rep = pd.DataFrame(rows); rep["p1"] = rep["p1"].astype(str); rep["p2"] = rep["p2"].astype(str)
    for a, b in [("S2", "S1"), ("S2", "S3"), ("S2", "S4"), ("S4", "S1")]:
        rep[f"{a}-{b}"] = rep[f"{a}_prec"] - rep[f"{b}_prec"]
else:
    rep = pd.read_csv(f"{REPO}/results/repeated_splits.csv")
for spec, g in rep.groupby("spec"):
    print(f"\n=== {spec} ===")
    print(g[["S1_prec", "S2_prec", "S3_prec", "S4_prec", "S2-S1", "S2-S3", "S2-S4", "S4-S1"]].agg(["mean", "std", "min", "max"]).T.round(4))
    print("S2 > S1:", int((g["S2-S1"] > 0).sum()), "| S2 > S3:", int((g["S2-S3"] > 0).sum()), "| S2 > S4:", int((g["S2-S4"] > 0).sum()), "of", len(g))
    print("cap chosen:", g["cap"].value_counts().to_dict(), "| w chosen:", g["blend_w"].value_counts().to_dict(), "| engine:", g["selected_engine"].value_counts().to_dict())

## 7. Class balancing and preprocessing isolation (Review 3 point 4)

In [ ]:
import inspect
src = inspect.getsource(R.xgb_pipeline) + inspect.getsource(R.tune_logit) + inspect.getsource(R.nested_run)
assert "scale_pos_weight=spw" in src and "(y == 0).sum() / (y == 1).sum()" in src, "positive-class weight must come from the training labels passed in"
assert 'class_weight="balanced"' in src
assert "high_missing_columns(X_tr)" in src, "missingness rule must be decided on training rows"
print("Class weighting is computed from training labels only; no resampling anywhere; >40%-missing rule applied on training rows of each split.")

## 8. Check against the stored results file

In [ ]:
S = json.load(open(f"{REPO}/results/summary.json"))
for spec in ["nosms", "orig"]:
    for k in ["S1_conv", "S2_conv", "S3_conv", "S4_conv"]:
        assert runs[spec][k] == S[f"primary_{spec}"][k], (spec, k, runs[spec][k], S[f"primary_{spec}"][k])
print("Primary-split converter counts match results/summary.json:", {s: [runs[s][k] for k in ["S1_conv", "S2_conv", "S3_conv", "S4_conv"]] for s in runs})
print("Provenance in summary.json:", {k: S["provenance"][k] for k in ["dataset_sha256", "xgboost", "sklearn", "bootstrap_n", "repeated_seeds"]})